In [ ]:
import os
from dotenv import load_dotenv

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings

C:\Users\Admin\AppData\Local\Temp\ipykernel_18188\4151740514.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [ ]:
load_dotenv

<function dotenv.main.load_dotenv(dotenv_path: Union[str, ForwardRef('os.PathLike[str]'), NoneType] = None, stream: Optional[IO[str]] = None, verbose: bool = False, override: bool = False, interpolate: bool = True, encoding: Optional[str] = 'utf-8') -> bool>

In [ ]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

In [ ]:
# load data
DATA_FILE_PATH = os.path.join("data", "hr_policy.txt")

In [ ]:
# Data Ingestion 
loader = TextLoader(DATA_FILE_PATH)
documents = loader.load()

In [ ]:
documents[0].page_content

'COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nDuring probation, employees are not eligible for paid leave, but may take unpaid 

In [ ]:
len(documents)

1

In [ ]:
documents[0].metadata

{'source': 'data\\hr_policy.txt'}

In [ ]:
# spliiting
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)

In [ ]:
chunks = text_splitter.split_documents(documents)

In [ ]:
chunks

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'),
 Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'),
 Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 

In [ ]:
len(chunks)

9

In [ ]:
chunks[2].page_content

'2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.'

In [ ]:
# embeddings
embeddings_model = JinaEmbeddings(model_name="jina-embeddings-v3")

In [ ]:
from langchain_community.vectorstores import FAISS
vector_store = FAISS.from_documents(chunks,embeddings_model)


In [ ]:
test_query = "How many sick leaves employees get"

## SIMILARITY SEARCH 

top_matches = vector_store.similarity_search(test_query , k=3)
print(f"Query: {test_query}\n")
for i,match in enumerate(top_matches,start=1):
    print(f"--- Match {i} ---")
    print(match.page_content)
    print()


Query: How many sick leaves employees get

--- Match 1 ---
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

--- Match 2 ---
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.

--- Match 3 ---
2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable du

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})  # returns top 3 relevant chunks

def search_hr_policy(question:str)->str:
    """
    Search the HR policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process.
    
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

In [ ]:
# data retrieval 

from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.2)

In [ ]:
find = llm.invoke("tell me about leave policy")

In [ ]:
find.content

'Below is a broad overview of what a typical **leave policy** looks like in many organizations.  The exact details will vary by country, state/province, industry, and the specific employer, but the categories and common rules are fairly consistent.\n\n---\n\n## 1. Core Types of Leave\n\n| Leave Type | Typical Purpose | Common Eligibility / Accrual Rules | Typical Duration |\n|------------|----------------|------------------------------------|------------------|\n| **Annual / Vacation Leave** | Rest, personal time, travel | Earned based on length of service (e.g., 1–2\u202fdays per month). Often prorated for new hires. | 10–30\u202fdays per year (varies widely) |\n| **Sick / Medical Leave** | Illness, injury, medical appointments | May accrue (e.g., 1\u202fday per month) or be granted as a fixed pool. Some jurisdictions require “unlimited” sick leave. | Usually 5–15\u202fdays per year; can be carried over or capped |\n| **Personal/Carer Leave** | Caring for a family member, personal eme